# Screaming Frog file maniuplation

Some of our shared folders (e.g. screaming frog output) contain data split across many numbered csv files with the same columns, e.g. `html_1.csv`, `html_2.csv`, ... or `pdf_1.csv`, `pdf_2.csv`, ...

In [9]:
import pandas as pd
import os
import pandas
import sqlalchemy
from io import BytesIO
from dwutils import db, s3

In [ ]:
### this is to create to files to run on screaming frog
# incremental updates in future should only need to run those that have changed (last updated date)
engine = sqlalchemy.create_engine('dialect://', execution_options={"stream_results": True})
df = pandas.read_sql(sqlalchemy.text("""SELECT public_url FROM \"example_catalogue_schema\".\"example_content_table_notaxons\""""), engine)
output_dir = "chunks"
os.makedirs(output_dir, exist_ok=True)

chunk_size = 10_000

for i, start in enumerate(range(0, len(df), chunk_size), start=1):
    chunk = df.iloc[start:start + chunk_size]

    with open(f"{output_dir}/chunk_{i:02d}.txt", "w") as f:
        f.write("\n".join(chunk["public_url"].astype(str)))

print(f"Created {i} files")

## Function definitions

In [2]:
def read_csv_from_shared_folder(
    file_name, path="path/to/crawl_data/", team="example_team", **kwargs
):
    """
    Wrapper function for reading csv files from s3 shared folder.
    """

    with s3.read(path=os.path.join(path, file_name), team=team) as f:
        data = pd.read_csv(f, **kwargs)
    return data

def generate_file_names(prefix, n_files, extension="csv", start=1):
    """
    Builds a list of numbered file names, e.g. prefix="html", n_files=16 ->
    ["html_1.csv", "html_2.csv", ..., "html_16.csv"]
    params:
        prefix: string. e.g. "html" or "pdf"
        n_files: int. how many numbered files to generate
        extension: string. file extension, without the dot
        start: int. the number to start counting from
    returns:
        list(str)
    """
    return [f"{prefix}{i}.{extension}" for i in range(start, start + n_files)]


def read_and_concat_csv_files(file_names, path, team="example_team", **kwargs):
    """
    Reads any number of csv files from a shared s3 folder and concatenates them together.
    Assumes all files have the same columns.
    Change `path` to point at a different folder (e.g. the html folder vs the pdf folder).
    params:
        file_names: list(str). names of the csv files to read, e.g. from generate_file_names
        path: string. the s3 folder the files live in
        team: string. the s3 team schema to read from
        **kwargs: passed through to pd.read_csv (e.g. dtype, parse_dates)
    returns:
        pd.DataFrame
    """
    data_list = [
        read_csv_from_shared_folder(file_name=file_name, path=path, team=team, **kwargs)
        for file_name in file_names
    ]
    return pd.concat(data_list, ignore_index=True)


def read_and_concat_to_from_columns(file_names, path, team="example_team", **kwargs):
    """
    Same as read_and_concat_csv_files, but keeps only the "To" and "From" columns.
    """
    data = read_and_concat_csv_files(file_names, path=path, team=team, **kwargs)
    return data[["To", "From"]]

## Usage

Change `path`, `prefix` and `n_files` to switch which folder/set of files you read from.

In [ ]:
# example: 9 html files produced by scraming frog e.g. html_1.csv ... html_16.csv
html_path_int = "path/to/crawl_data/html_internal/"
#html_files = generate_file_names(prefix="html_internal", n_files=9)
html_files_full = generate_file_names(prefix="html_full", n_files=9)

#html_data = read_and_concat_csv_files(html_files, path=html_path_int)
html_to_from = read_and_concat_to_from_columns(html_files_full, path=html_path_int)

In [15]:
with s3.write(team="example_team", path="path/to/crawl_data/html_internal/html_int_data.csv",
    mode="string"
) as f:
    html_data.to_csv(f, index=False)
with s3.write(team="example_team", path="path/to/crawl_data/html_internal/all_int_html.csv",
    mode="string"
) as f:
    html_to_from.to_csv(f, index=False)

In [6]:
# external 
all_html = read_csv_from_shared_folder(file_name='all_html.csv', path="path/to/crawl_data/html_external/", team='example_team')
all_html = all_html[["To", "From"]]
with s3.write(team="example_team", path="path/to/crawl_data/html_external/all_html.csv",
    mode="string"
) as f:
    all_html.to_csv(f, index=False)

In [ ]:
pdf_path = "path/to/crawl_data/pdf/"
pdf_files = generate_file_names(prefix="pdf_", n_files=8)
pdf_data = read_and_concat_csv_files(pdf_files, path=pdf_path)

#pdf_to_from = read_and_concat_to_from_columns(pdf_files, path=pdf_path)

In [6]:
with s3.write(team="example_team", path="path/to/crawl_data/pdf/all_pdfs_data.csv",
    mode="string"
) as f:
    pdf_data.to_csv(f, index=False)
    

In [ ]:
# you will need to do a join to main (old) data in tool and any matches need to keep the new data from screaming frog
# this then gets matched as previously to GA

# Combining a zip of text files into a single csv

Shared folder contains a zip archive of thousands of individual `text files (one per page), where each file's name encodes the page's pdf url and its contents are the extracted page text.

The functions below unzip that archive straight from s3 and combine every text file into a single dataframe/csv with two columns: `url` and `pdf_clean`. The text column is truncated to a maximum of 3,000 words per page to keep the output a manageable size.


In [10]:
import zipfile

def truncate_words(text, max_words=3000):
    words = text.split()
    return " ".join(words[:max_words])


def read_zip_text_files_to_dataframe(
    zip_file_name,
    path,
    team="example_team",
    max_words=5000,
    encoding="utf-8",
):
    with s3.read(path=os.path.join(path, zip_file_name), team=team) as f:
        zip_bytes = BytesIO(f.read())

    rows = []
    skipped_binary_pdfs = []
    with zipfile.ZipFile(zip_bytes) as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue
            with zf.open(info) as file_obj:
                raw = file_obj.read()
            # some files in these archives are named *.pdf even though their
            # contents are plain text; only skip ones that are genuinely
            # binary pdfs (real pdf magic bytes), so nothing text-based is lost
            if raw.startswith(b"%PDF-"):
                skipped_binary_pdfs.append(info.filename)
                continue
            text = raw.decode(encoding, errors="replace")
            rows.append(
                {
                    "url": info.filename,
                    "pdf_clean": truncate_words(text, max_words=max_words),
                }
            )

    if skipped_binary_pdfs:
        print(f"Skipped {len(skipped_binary_pdfs)} genuine binary pdf file(s) (not text):")
        for name in skipped_binary_pdfs:
            print(f"  {name}")

    return pd.DataFrame(rows, columns=["url", "pdf_clean"])

In [12]:
text_zip_path = "path/to/crawl_data/pdf/"
text_zip_file = "pdf_texts.zip"

page_text_data = read_zip_text_files_to_dataframe(
    zip_file_name=text_zip_file, path=text_zip_path
)

In [16]:
with s3.write(
    team="example_team",
    path=os.path.join(text_zip_path, "pdf_full_text_dataframe.csv"),
    mode="string",
) as f:
    page_text_data.to_csv(f, index=False)